## 介紹
本程式使玉pyautogui來完成一個Blitz的遊戲機器人。

In [1]:
import pyautogui as pyg
pyg.sleep(2)
basex=282
basey=382
gap=40


In [2]:
def get_gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心的 (x, y) 座標，索引從 0 開始。"""
    return basex + c * gap, basey + r * gap

In [3]:
def get_gem_color(r, c, im):
    """讀取截圖中第 r 列、第 c 欄寶石中心的顏色。"""
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [4]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

(282, 382)
(322, 382)
(282, 422)


In [ ]:
def _can_move(r, c, im, dr, dc):
    """交換至指定方向後，檢查是否會形成三連線。"""
    nr, nc = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    target_color = get_gem_color(nr, nc, im)
    if current_color == target_color:
        return False

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return target_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for row_step, col_step in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                test_r = row + direction * row_step
                test_c = col + direction * col_step
                while (0 <= test_r < 8 and 0 <= test_c < 8
                       and color_after_swap(test_r, test_c) == color):
                    count += 1
                    test_r += direction * row_step
                    test_c += direction * col_step
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_left(r, c, im):
    return _can_move(r, c, im, 0, -1)

def can_move_right(r, c, im):
    return _can_move(r, c, im, 0, 1)

def can_move_up(r, c, im):
    return _can_move(r, c, im, -1, 0)

def can_move_down(r, c, im):
    return _can_move(r, c, im, 1, 0)

In [6]:
im=pyg.screenshot()
for i  in range(8):
    x,y=get_gem_position(0,i)
    pyg.click(x,y,duration=1)
    print(get_gem_color(0,i,im))

(253, 28, 57)
(18, 137, 251)
(252, 234, 107)
(252, 234, 107)
(237, 15, 237)
(251, 251, 251)
(251, 251, 251)
(252, 234, 107)


In [ ]:
import time

start_time = time.monotonic()
im = pyg.screenshot()
while True:
    if time.monotonic() - start_time > 60:
        break

    moved = False
    for r in range(8):
        for c in range(8):
            directions = (
                (can_move_left, 0, -1),
                (can_move_right, 0, 1),
                (can_move_up, -1, 0),
                (can_move_down, 1, 0),
            )
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    start_x, start_y = get_gem_position(r, c)
                    end_x, end_y = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(start_x, start_y)
                    pyg.dragTo(end_x, end_y, duration=0.15)
                    moved = True
                    break
            if moved:
                break
        if moved:
            break

    if moved:
        pyg.sleep(0.2)
    else:
        pyg.sleep(0.05)
    im = pyg.screenshot()